<p style="text-align:center">
    <a href="https://skills.network/?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMML321ENSkillsNetwork817-2022-01-01" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Extract Bag of Words (BoW) Features from Course Textual Content**


Estimated time needed: **60** minutes


## **Introduction**

In this lab, we will perform **feature engineering using the Bag-of-Words (BoW)** method to convert course descriptions into numerical features that can be used in machine learning models.

The course dataset already contains **genre information** for each course.
However, genre labels are usually very general, such as *Data Science*, *Business*, or *Cloud Computing*. Courses that belong to the same genre may still be very different in their actual content. Therefore, genre information alone is not enough to describe the course in detail.

Each course also has a **text description**, which contains more specific information about the topics covered in the course. To use this information in machine learning models, we need to convert the text into numerical features.

One common method is the **Bag-of-Words (BoW)** model, which represents each course description as a vector of word counts. Each unique word becomes a feature, and the value indicates how often the word appears in the description. This allows us to capture more detailed differences between courses than using genre alone.

![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_2/images/extract_textual_features.png)


In this lab, we will extract BoW features from course text and prepare them for use in later recommendation system tasks.

## **Objectives**

* Extract Bag of Words (BoW) features from course titles and descriptions
* Build a course BoW dataset for later similarity computation and a content-based recommender system

----


### **Prepare and setup the lab environment**


First, let's install and import required libraries:


In [ ]:
# !pip install nltk==3.6.7
# !pip install gensim
# !pip install scipy==1.10
# !pip install pandas
# !pip install nltk
# !pip install matplotlib

In [ ]:
import gensim
import pandas as pd
import nltk as nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora

%matplotlib inline

Download stopwords


In [ ]:
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

In [ ]:
# also set a random state
rs = 123

### **Extract bag of words (BoW) features from exsample courses**


BoW features are essentially the counts or frequencies of each word that appears in a text (string). Let's illustrate it with some simple examples.


Suppose we have two course descriptions as follows:


In [ ]:
course1 = "this is an introduction data science course which introduces data science to beginners"

In [ ]:
course2 = "machine learning for beginners"

In [ ]:
courses = [course1, course2]
courses

#### **Tokenize course text data**

The first step is to split the two strings into words (tokens). A token in the text processing context means the smallest unit of text such as a word, a symbol/punctuation, or a phrase, etc. The process to transform a string into a collection of tokens is called `tokenization`.


One common way to do ```tokenization``` is to use the Python built-in `split()` method of the `str` class.  However, in this lab, we want to leverage the `nltk` (Natural Language Toolkit) package, which is probably the most commonly used package to process text or natural language.


 More specifically, we will use the ```word_tokenize()``` method on the content of course (string):


In [ ]:
# Tokenize the two courses
tokenized_courses = [word_tokenize(course) for course in courses]
# tokenized_courses = [course.split() for course in courses]

In [ ]:
tokenized_courses[:20]

In the cell output, two courses have been tokenized and turned into two token arrays.


#### **Build a vocabulary of words**

Next, we want to create a token dictionary to index all tokens. Basically, we want to assign a key/index for each token. One way to index tokens is to use the `gensim` package which is another popular package for processing textual data:


In [ ]:
# Create a token dictionary for the two courses
tokens_dict = gensim.corpora.Dictionary(tokenized_courses)

In [ ]:
print(tokens_dict.token2id)

#### **Create Bag-of-Words vectors**

With the token dictionary, we can easily count each token in the two example courses and output two BoW feature vectors. However, more conveniently, the `gensim` package provides us a `doc2bow` method to generate BoW features out-of-box.


In [ ]:
# Generate BoW features for each course
courses_bow = [tokens_dict.doc2bow(course) for course in tokenized_courses]

In [ ]:
courses_bow

It outputs two BoW arrays where each element is a tuple, e.g., (0, 1) and (7, 2). The first element of the tuple is the token ID and the second element is its count. So `(0, 1)` means `(``an``, 1)` and `(7, 2)` means `(``science``, 2)`.


We can use the following code snippet to print each token and its count:


In [ ]:
# Enumerate through each course and its bag-of-words representation
for course_idx, course_bow in enumerate(courses_bow):
    # Print the index of the current course and a label
    print(f"Bag of words for course {course_idx}:")
    # For each token index, print its bow value (word count)
    for token_index, token_bow in course_bow:
        # Retrieve the token from the tokens dictionary based on its index
        token = tokens_dict.get(token_index)
        # Print the token and its bag-of-words value
        print(f"--Token: '{token}', Count:{token_bow}")

In [ ]:
# another way to counts of words in a document
tokens_dict = gensim.corpora.Dictionary(tokenized_courses)

for i, course in enumerate(tokenized_courses):
    print(f"Bag of words for course {i}:")
    for token_id, count in tokens_dict.doc2bow(course):
        print(f"--Token: '{tokens_dict[token_id]}', Count:{count}")

If we turn to the long list into a horizontal feature vectors, we can see the two courses become two numerical feature vectors:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/module_2/images/bow.png)


### **BoW dimensionality reduction**


A document may contain tens of thousands of words which makes the dimension of the BoW feature vector huge. To reduce the dimensionality, one common way is to filter the relatively meaningless tokens such as stop words or sometimes add position and adjective words.


We can use the english stop words provided in `nltk`:


In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:
list(stop_words)[:10]

Then we can filter those English stop words from the tokens in course1:


In [ ]:
# Tokens in course 1
tokenized_courses[0]

In [ ]:
processed_tokens = [w for w in tokenized_courses[0] if not w.lower() in stop_words]

In [ ]:
processed_tokens

The number of tokens for ```course1``` has been reduced, after stop words being removed.


Another common way is to only keep nouns in the text. We can use the `nltk.pos_tag()` method to analyze the part of speech (POS) and annotate each word.


In [ ]:
tags = nltk.pos_tag(tokenized_courses[0])
tags

As we can see [`introduction`, `data`, `science`, `course`, `beginners`] are all of the nouns and we may keep them in the BoW feature vector.


### **Extract BoW features for course textual content and build a dataset**


In [ ]:
course_url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML321EN-SkillsNetwork/labs/datasets/course_processed.csv"
course_content_df = pd.read_csv(course_url)

In [ ]:
course_content_df.head(5)

In [ ]:
course_content_df.iloc[0, :]

The course content dataset has three columns `COURSE_ID`, `TITLE`, and `DESCRIPTION`. `TITLE` and `DESCRIPTION` are all text upon which we want to extract BoW features. 


Let's join those two text columns together.


In [ ]:
# Merge TITLE and DESCRIPTION title
course_content_df['course_texts'] = course_content_df[['TITLE', 'DESCRIPTION']].agg(' '.join, axis=1)
course_content_df = course_content_df.reset_index()

In [ ]:
course_content_df.head()

In [ ]:
course_content_df.iloc[0, :]

#### **Custom tokenizer/cleaner of course text data**

We create the `tokenize_course()` method  to tokenize the course content:


In [ ]:
def tokenize_course(course, keep_only_nouns=True):
    # Get English stop words
    stop_words = set(stopwords.words('english'))
    # Tokenize the course text
    word_tokens = word_tokenize(course)
    # Remove English stop words and numbers
    word_tokens = [w for w in word_tokens if (not w.lower() in stop_words) and (not w.isnumeric())]
    # Only keep nouns 
    if keep_only_nouns:
        # Define a filter list of non-noun POS tags
        filter_list = ['WDT', 'WP', 'WRB', 'FW', 'IN', 'JJR', 'JJS', 'MD', 'PDT', 'POS', 'PRP', 'RB', 'RBR', 'RBS',
                       'RP']
        # Tag the word tokens with POS tags
        tags = nltk.pos_tag(word_tokens)
        # Filter out non-nouns based on POS tags
        word_tokens = [word for word, pos in tags if pos not in filter_list]

    return word_tokens

Let's try it on the first course.


In [ ]:
a_course = course_content_df.iloc[0, :]['course_texts']
a_course

In [ ]:
tokenize_course(a_course)

#### **Implementation Option 1: Use Gensim library**

##### **1. Tokenzing all courses in the `courses_df`**


_TODO: Use provided tokenize_course() method to tokenize all courses in courses_df['course_texts']._


In [ ]:
course_content_df['tokenized_course_texts'] = (
    course_content_df['course_texts'].apply(tokenize_course)
)
course_content_df.head(5)


<details>
    <summary>Click here for Hints</summary>

Use `tokenize_course(text, True)` command to tokenize each text in `courses_df['course_texts']`


##### **2. Build a vocabulary of words with `tokens_dict`**

Then we need to create a token dictionary `tokens_dict`

_TODO: Use gensim.corpora.Dictionary(tokenized_courses) to create a token dictionary._


In [ ]:
tokens_dict = gensim.corpora.Dictionary(course_content_df['tokenized_course_texts'])
list(tokens_dict.token2id.items())[:20]

##### **3. Create Bag-of-Words vectors with `doc2bow()`**

Then we can use `doc2bow()` method to generate BoW features for each tokenized course.


_TODO: Use tokens_dict.doc2bow() to generate BoW features for each tokenized course._


In [ ]:
for i, course in enumerate(course_content_df['tokenized_course_texts']):
    print(f"Bag of words for course {i}:")
    for token_id, count in tokens_dict.doc2bow(course):
        print(f"--Token: '{tokens_dict[token_id]}', Count:{count}")

<details>
    <summary>Click here for Hints</summary>
    
You can use `tokens_dict.doc2bow(course)` command  for each course in `tokenized_courses`


##### **4. Store the results in a structured format for later use**

Lastly, you need to append the BoW features for each course into a new BoW dataframe. The new dataframe needs to include the following columns (you may include other relevant columns as well):
- 'doc_index': the course index starting from 0
- 'doc_id': the actual course id such as `ML0201EN`
- 'token': the tokens for each course
- 'bow': the bow value for each token


_TODO: Create a new course_bow dataframe based on the extracted BoW features._


In [ ]:
bow_rows = []

for i, course in enumerate(course_content_df['tokenized_course_texts']):
    doc_id = course_content_df['COURSE_ID'][i]   # example: ML0201EN

    for token_id, count in tokens_dict.doc2bow(course):
        bow_rows.append({
            'doc_index': i,
            'doc_id': doc_id,
            'token': tokens_dict[token_id],
            'bow': count
        })

bow_df = pd.DataFrame(bow_rows)

bow_df.head()
#  ...
#  bow_dicts = {"doc_index": doc_indices,
#            "doc_id": doc_ids,
#            "token": tokens,
#            "bow": bow_values}
#  pd.DataFrame(bow_dicts)

<details>
    <summary>Click here for Hints</summary>
    
You can use 2 for-loops to create your data frame: first one will be `for doc_index, doc_bow in enumerate(bow_docs):` where bow_docs is the list of BoW features for each tokenized course and within this for-loop you will have another loop `for token_index, token_bow in doc_bow:`. Then you can get each "token" by applying the `token_index` to your `token_dict`,  `token_bow` will give you "bow" values, `doc_indices` will give you values for  "doc_index" and you can get "doc_id" by using `courses_df['COURSE_ID']` list and `doc_index` as indexes.


#### **Implementation Option 2: Use scikit-learn library**

##### **1. BoW builder using `CountVectorizer`**

`CountVectorizer` build vocabulary from ALL documents, and then will create matrix with ALL words for EACH document. So it produce tokens with zero count. 
`Gensim` dictionary + `doc2bow` returns a list of pairs, and only stores tokens what exist.

In [ ]:
import sklearn
from sklearn.feature_extraction.text import CountVectorizer

cv = CountVectorizer(
    tokenizer=lambda text: tokenize_course(text, keep_only_nouns=True),
    lowercase=False,
    preprocessor=None
)

bow_mat = cv.fit_transform(course_content_df["course_texts"])


##### **2. Convert to dataframe (doc_index, doc_id, token, bow)**

In [ ]:
bow_df_2 = (
    pd.DataFrame.sparse.from_spmatrix(
        bow_mat,
        index=course_content_df.index,
        columns=cv.get_feature_names_out()
    )
    .reset_index()
    .rename(columns={"index": "doc_index"})
)

bow_df_2.insert(1, "doc_id", course_content_df["COURSE_ID"].values)

bow_df_2 = bow_df_2.melt(
    id_vars=["doc_index", "doc_id"],
    var_name="token",
    value_name="bow"
)

# keep only the non-zero word counts and clean the index
bow_df_2 = bow_df_2[bow_df_2["bow"] > 0].reset_index(drop=True)

bow_df_2[bow_df_2['doc_id']== 'ML0201EN'].head()

### **Other popular textual features**


In addition to the basic token BoW feature, there are two other types of widely used textual features. 

- **tf-idf**: tf-idf refers to Term Frequency–Inverse Document Frequency. Similar to BoW, the tf-idf also counts the word frequencies in each document. Furthermore, tf-idf will  offset the number of documents in the corpus that contain the word in order to adjust for the fact that some words appear more frequently in general. The higher the tf-idf normally means the greater the importance the word/token is.
- **Text embedding vector**. Embedding means projecting an object into a latent feature space. We normally employ neural networks or deep neural networks to learn the latent features of a textual object such as a word, a sentence, or the entire document. The learned latent feature vectors will be used to represent the original textual entities. 


#### **Term Frequency - Inverse Document Frequency (tf-idf)**
We explore how to use tf-idf to extract useful words from the course textual content.

##### **BoW vs tf-idf**
Example — 3 course descriptions. 
- Doc1: machine learning course 
- Doc2: machine learning basics 
- Doc3: cloud computing course

**BoW** counts how many times each word appears.
| word      | doc1 | doc2 | doc3 |
| --------- | ---- | ---- | ---- |
| machine   | 1    | 1    | 0    |
| learning  | 1    | 1    | 0    |
| course    | 1    | 0    | 1    |
| basics    | 0    | 1    | 0    |
| cloud     | 0    | 0    | 1    |
| computing | 0    | 0    | 1    |


`course` and `learn` appear in almost all documents. BoW thinks they are important, but they are not. 

**TF-IDF** means: Important words = words that appear a lot in this document but NOT in all documents \
- score = word_count × how rare the word is 
- common words → small value 
- rare words → big value
| word      | doc1 | doc2 | doc3 |
| --------- | ---- | ---- | ---- |
| machine   | 0.5  | 0.5  | 0    |
| learning  | 0.5  | 0.5  | 0    |
| course    | 0.2  | 0    | 0.2  |
| basics    | 0    | 0.8  | 0    |
| cloud     | 0    | 0    | 0.9  |
| computing | 0    | 0    | 0.9  |

`course` → small weight, 
`machine` → medium, 
`cloud` → high

TF-IDF is usually used instead of BoW for ML/recommender

In [ ]:
import re
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
# Count Vectorizer creates a term frequency matrix
pd.DataFrame(bow_mat.toarray(), columns = cv.get_feature_names_out())

In [ ]:
# Creating the tfidf matrix
tfidf_trans = TfidfTransformer(smooth_idf=False)
tfidf_mat = tfidf_trans.fit_transform(bow_mat)
tfidf = pd.DataFrame(tfidf_mat.toarray(), columns = cv.get_feature_names_out())
tfidf.head()

In [ ]:
tf_idf = course_content_df[['COURSE_ID']].join(tfidf)

tf_idf_long = pd.melt(
    tf_idf,
    id_vars=["COURSE_ID"],
    var_name="token",
    value_name="tf_idf",
    ignore_index=False
)

# keep only the non-zero word counts and clean the index
tf_idf_long = tf_idf_long[tf_idf_long["tf_idf"] > 0].reset_index(drop=True)

tf_idf_long[tf_idf_long['COURSE_ID']== 'ML0201EN'].head()

## **Summary**


In this lab, we have practiced extracting BoW features from course titles and descriptions. Once the feature vectors on the courses has been built, we can then apply machine learning algorithms such as similarity measurements, clustering, or classification on the courses in later labs.


## **Authors**


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/), Su Wu


Copyright © 2021 IBM Corporation. All rights reserved.
